In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import joblib

In [2]:
df = pd.read_csv("../data/paysim_engineered.csv")

In [3]:
df.info()
df.shape
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 19 columns):
 #   Column                            Dtype  
---  ------                            -----  
 0   step                              int64  
 1   amount                            float64
 2   oldbalanceOrg                     float64
 3   newbalanceOrig                    float64
 4   oldbalanceDest                    float64
 5   newbalanceDest                    float64
 6   isFraud                           int64  
 7   isFlaggedFraud                    int64  
 8   origin_balance_change             float64
 9   destination_balance_change        float64
 10  origin_balance_removed_pct        float64
 11  destination_balance_increase_pct  float64
 12  origin_balance_error              float64
 13  destination_balance_error         float64
 14  high_value_transaction            int64  
 15  type_CASH_OUT                     int64  
 16  type_DEBIT                        in

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,origin_balance_change,destination_balance_change,origin_balance_removed_pct,destination_balance_increase_pct,origin_balance_error,destination_balance_error,high_value_transaction,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,9839.64,170136.0,160296.36,0.0,0.0,0,0,9839.64,0.0,0.057834,0.0,0.0,9839.64,0,0,0,1,0
1,1,1864.28,21249.0,19384.72,0.0,0.0,0,0,1864.28,0.0,0.087735,0.0,0.0,1864.28,0,0,0,1,0
2,1,181.00,181.0,0.00,0.0,0.0,1,0,181.00,0.0,1.000000,0.0,0.0,181.00,0,0,0,0,1
3,1,181.00,181.0,0.00,21182.0,0.0,1,0,181.00,-21182.0,1.000000,-1.0,0.0,21363.00,0,1,0,0,0
4,1,11668.14,41554.0,29885.86,0.0,0.0,0,0,11668.14,0.0,0.280795,0.0,0.0,11668.14,0,0,0,1,0


In [4]:
#Seprates target and features
X = df.drop(columns="isFraud")

y = df["isFraud"]

In [5]:
y.value_counts()

isFraud
0    6354407
1       8213
Name: count, dtype: int64

In [6]:
negative = y.value_counts()[0]
positive = y.value_counts()[1]

scale_pos_weight = negative / positive
print(f"Scale Pos Weight: {scale_pos_weight:.2f}")

Scale Pos Weight: 773.70


In [7]:
#Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [8]:
print("Training Set Distribution")
print(y_train.value_counts())
print()
print(y_train.value_counts(normalize=True))

print("\n------------------------\n")

print("Testing Set Distribution")
print(y_test.value_counts())
print()
print(y_test.value_counts(normalize=True))

Training Set Distribution
isFraud
0    5083526
1       6570
Name: count, dtype: int64

isFraud
0    0.998709
1    0.001291
Name: proportion, dtype: float64

------------------------

Testing Set Distribution
isFraud
0    1270881
1       1643
Name: count, dtype: int64

isFraud
0    0.998709
1    0.001291
Name: proportion, dtype: float64


In [9]:
#Scale data
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [10]:
#Logistsic Regression
log_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

log_model.fit(
    X_train_scaled,
    y_train
)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [11]:
#Predictions
log_predictions = log_model.predict(X_test_scaled)
log_probabilities = log_model.predict_proba(X_test_scaled)[:, 1]

In [12]:
#Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train,
    y_train
)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [13]:
#Predictions
rf_predictions = rf_model.predict(X_test)
rf_probabilities = rf_model.predict_proba(X_test)[:,1]

In [14]:
#XGBoost
xgb_model = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight
)

xgb_model.fit(
    X_train,
    y_train
)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [15]:
#Predictions
xgb_predictions = xgb_model.predict(X_test)
xgb_probabilities = xgb_model.predict_proba(X_test)[:,1]

In [16]:
#Save
joblib.dump(log_model, "../results/logistic_model.pkl")

joblib.dump(rf_model, "../results/random_forest_model.pkl")

joblib.dump(xgb_model, "../results/xgboost_model.pkl")

joblib.dump(
    scaler,
    "../results/scaler.pkl"
)

joblib.dump(
    (X_train, X_test, y_train, y_test),
    "../results/train_test_split.pkl"
)

['../results/train_test_split.pkl']

In [17]:
#save
results = pd.DataFrame({
    "Actual": y_test,
    "Logistic": log_predictions,
    "RandomForest": rf_predictions,
    "XGBoost": xgb_predictions
})

results.to_csv(
    "../results/model_predictions.csv",
    index=False
)